In [ ]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import LabelEncoder

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_context("talk")

WORK_DIR = "/kaggle/working"
CLEAN_PATH = os.path.join(WORK_DIR, "cleaned_dataset.parquet")

df = pd.read_parquet(CLEAN_PATH)
print("Loaded cleaned dataset:", df.shape)
df.head()

In [ ]:
print("DataFrame info:")
df.info()

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print("\nNumber of numeric columns:", len(numeric_cols))

In [ ]:
# We'll compute an approximate entropy over packet length distribution
# using Packet Length Min/Max/Mean/Std.

packet_len_cols = [
    "Packet Length Min",
    "Packet Length Max",
    "Packet Length Mean",
    "Packet Length Std",
]

for col in packet_len_cols:
    if col not in df.columns:
        print(f"Warning: {col} not in df.")

def approximate_flow_entropy(row, eps=1e-9):
    # construct a simple 4-bin distribution from basic stats
    vals = np.array([
        max(row["Packet Length Min"], 0),
        max(row["Packet Length Mean"] - row["Packet Length Std"], 0),
        max(row["Packet Length Mean"] + row["Packet Length Std"], 0),
        max(row["Packet Length Max"], 0),
    ], dtype=float)
    if np.any(np.isnan(vals)) or np.all(vals == 0):
        return 0.0
    probs = vals / (vals.sum() + eps)
    return -(probs * np.log2(probs + eps)).sum()

df["flow_entropy"] = df[packet_len_cols].apply(approximate_flow_entropy, axis=1)
df["flow_entropy"].describe()

In [ ]:
# Replace with actual column names if different
SRC_IP_COL = "Src IP"   # or "Source IP"
DST_IP_COL = "Dst IP"   # or "Destination IP"
SRC_PORT_COL = "Src Port"
DST_PORT_COL = "Dst Port"

missing_ip_cols = [c for c in [SRC_IP_COL, DST_IP_COL, SRC_PORT_COL, DST_PORT_COL] if c not in df.columns]
print("Missing IP/port cols:", missing_ip_cols)

In [ ]:
if not missing_ip_cols:
    # Unique destination IPs per source IP
    src_to_dst_unique = (
        df.groupby(SRC_IP_COL)[DST_IP_COL]
          .nunique()
          .rename("src_unique_dst_ip")
    )

    # Unique destination ports per source IP
    src_to_dst_ports_unique = (
        df.groupby(SRC_IP_COL)[DST_PORT_COL]
          .nunique()
          .rename("src_unique_dst_port")
    )

    df = df.join(src_to_dst_unique, on=SRC_IP_COL)
    df = df.join(src_to_dst_ports_unique, on=SRC_IP_COL)
else:
    # If you don't have IP/port columns in this parquet, set features to NaN
    df["src_unique_dst_ip"] = np.nan
    df["src_unique_dst_port"] = np.nan

df[["src_unique_dst_ip", "src_unique_dst_port"]].describe()

In [ ]:
# Columns for temporal stats (CICIDS2017 standard names)
FLOW_DURATION_COL = "Flow Duration"
FLOW_IAT_MEAN_COL = "Flow IAT Mean"
FLOW_IAT_STD_COL = "Flow IAT Std"

for col in [FLOW_DURATION_COL, FLOW_IAT_MEAN_COL, FLOW_IAT_STD_COL]:
    if col not in df.columns:
        print("Missing temporal column:", col)

# Inter-arrival variance
df["flow_iat_var"] = df[FLOW_IAT_STD_COL] ** 2

# Burstiness: (sigma - mu) / (sigma + mu)
def compute_burstiness(row, eps=1e-9):
    mu = row[FLOW_IAT_MEAN_COL]
    sigma = row[FLOW_IAT_STD_COL]
    if mu < 0 or sigma < 0:
        return 0.0
    denom = sigma + mu + eps
    return (sigma - mu) / denom

df["flow_burstiness"] = df[[FLOW_IAT_MEAN_COL, FLOW_IAT_STD_COL]].apply(compute_burstiness, axis=1)
df[["flow_iat_var", "flow_burstiness"]].describe()

In [ ]:
from scipy.stats import skew, kurtosis

# Pick a small, meaningful subset of numeric features for per-flow stats
stat_feature_group = [
    "Total Fwd Packets",
    "Total Backward Packets",
    "Flow Bytes/s",
    "Flow Packets/s",
    "Packet Length Mean",
    "Packet Length Std",
]

missing_stat_cols = [c for c in stat_feature_group if c not in df.columns]
print("Missing stat cols:", missing_stat_cols)

In [ ]:
def row_skewness(row):
    vals = row.values.astype(float)
    if np.any(np.isnan(vals)):
        vals = vals[~np.isnan(vals)]
    if len(vals) < 3:
        return 0.0
    return float(skew(vals, bias=False))

def row_kurtosis(row):
    vals = row.values.astype(float)
    if np.any(np.isnan(vals)):
        vals = vals[~np.isnan(vals)]
    if len(vals) < 4:
        return 0.0
    return float(kurtosis(vals, fisher=True, bias=False))

df["flow_stat_skewness"] = df[stat_feature_group].apply(row_skewness, axis=1)
df["flow_stat_kurtosis"] = df[stat_feature_group].apply(row_kurtosis, axis=1)

df[["flow_stat_skewness", "flow_stat_kurtosis"]].describe()

In [ ]:
engineered_cols = [
    "flow_entropy",
    "flow_iat_var",
    "flow_burstiness",
    "flow_stat_skewness",
    "flow_stat_kurtosis",
    "src_unique_dst_ip",
    "src_unique_dst_port",
]

all_feature_cols = [c for c in numeric_cols] + engineered_cols
all_feature_cols = [c for c in all_feature_cols if c in df.columns]

print("Total feature columns (including engineered):", len(all_feature_cols))

In [ ]:
target_col = "attack_category"
le = LabelEncoder()
y = le.fit_transform(df[target_col])

# For MI, work on a sample to keep runtime manageable
MI_SAMPLE_SIZE = 500_000
if len(df) > MI_SAMPLE_SIZE:
    df_mi = df.sample(MI_SAMPLE_SIZE, random_state=42)
else:
    df_mi = df.copy()

X_mi = df_mi[all_feature_cols].fillna(0).astype(float)
y_mi = le.transform(df_mi[target_col])

mi_scores = mutual_info_classif(X_mi, y_mi, random_state=42)
mi_series = pd.Series(mi_scores, index=all_feature_cols).sort_values(ascending=False)

mi_series.head(30)

In [ ]:
TOP_K_MI = 40

plt.figure(figsize=(10, 10))
sns.barplot(
    x=mi_series.head(TOP_K_MI),
    y=mi_series.head(TOP_K_MI).index,
    palette="viridis"
)
plt.title("Top mutual information features (including engineered)")
plt.xlabel("Mutual information")
plt.ylabel("Feature")
plt.tight_layout()
plt.savefig("mi_feature_ranking_with_engineered.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
mi_series.to_frame(name="mutual_information").reset_index().rename(columns={"index": "feature"}) \
         .to_csv("feature_ranking_mi.csv", index=False)

In [ ]:
# Placeholder: XGBoost feature importance will be computed in 04_Baseline_ML.ipynb
# using the same all_feature_cols list and then exported as xgboost_feature_importance.csv

# Placeholder: SHAP values will be computed for the best tree-based model
# and aggregated per feature into shap_feature_importance.csv

In [ ]:
TOP_SELECTED = 60  # adjust as needed based on MI curve

selected_features = mi_series.head(TOP_SELECTED).index.tolist()
len(selected_features), selected_features[:10]

In [ ]:
df_selected = df[selected_features + [target_col]]
df_selected.to_parquet(os.path.join(WORK_DIR, "selected_features.parquet"), index=False)
print("Saved selected_features.parquet with shape:", df_selected.shape)

In [ ]:
pd.Series(selected_features, name="feature").to_csv("selected_feature_list.csv", index=False)